<h2 style='color:purple' align='center'>Naive Bayes Tutorial Part 1: Predicting survival from titanic crash</h2>

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [27]:
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Name,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female,35.0,1,0,113803,53.1000,C123,S,1
4,5,"Allen, Mr. William Henry",3,male,35.0,0,0,373450,8.0500,NaN,S,0


In [28]:
df.drop(['PassengerId','Name','SibSp','Parch','Ticket','Cabin','Embarked'],axis='columns',inplace=True)
df.head()

,Pclass,Sex,Age,Fare,Survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0


In [29]:
inputs = df.drop('Survived',axis='columns')
target = df.Survived

In [30]:
#inputs.Sex = inputs.Sex.map({'male': 1, 'female': 2})

In [31]:
dummies = pd.get_dummies(inputs.Sex)
dummies.head(3)

,female,male
0,False,True
1,True,False
2,True,False


In [32]:
inputs = pd.concat([inputs,dummies],axis='columns')
inputs.head(3)

,Pclass,Sex,Age,Fare,female,male
0,3,male,22.0,7.2500,False,True
1,1,female,38.0,71.2833,True,False
2,3,female,26.0,7.9250,True,False


**I am dropping male column as well because of dummy variable trap theory. One column is enough to repressent male vs female**

In [33]:
inputs.drop(['Sex','male'],axis='columns',inplace=True)
inputs.head(3)

,Pclass,Age,Fare,female
0,3,22.0,7.2500,False
1,1,38.0,71.2833,True
2,3,26.0,7.9250,True


In [34]:
inputs.columns[inputs.isna().any()]

Index(['Age'], dtype='str')

In [35]:
inputs.Age[:10]

0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
5     NaN
6    54.0
7     2.0
8    27.0
9    14.0
Name: Age, dtype: float64

In [36]:
inputs.Age = inputs.Age.fillna(inputs.Age.mean())
inputs.head()

,Pclass,Age,Fare,female
0,3,22.0,7.2500,False
1,1,38.0,71.2833,True
2,3,26.0,7.9250,True
3,1,35.0,53.1000,True
4,3,35.0,8.0500,False


In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(inputs,target,test_size=0.3)

In [38]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()

In [39]:
model.fit(X_train,y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [40]:
model.score(X_test,y_test)

0.746268656716418

In [41]:
X_test[0:10]

,Pclass,Age,Fare,female
268,1,58.0,153.4625,True
482,3,50.0,8.0500,False
96,1,71.0,34.6542,False
72,2,21.0,73.5000,False
745,1,70.0,71.0000,False
144,2,18.0,11.5000,False
747,2,30.0,13.0000,True
412,1,33.0,90.0000,True
769,3,32.0,8.3625,False
829,1,62.0,80.0000,True


In [42]:
y_test[0:10]

268    1
482    0
96     0
72     0
745    0
144    0
747    1
412    1
769    0
829    1
Name: Survived, dtype: int64

In [43]:
model.predict(X_test[0:10])

array([1, 0, 1, 0, 1, 0, 1, 1, 0, 1])

In [44]:
model.predict_proba(X_test[:10])

array([[6.50614805e-05, 9.99934939e-01],
       [9.62117038e-01, 3.78829616e-02],
       [4.41359741e-01, 5.58640259e-01],
       [7.54925555e-01, 2.45074445e-01],
       [2.32466079e-01, 7.67533921e-01],
       [9.12476741e-01, 8.75232588e-02],
       [3.18711927e-01, 6.81288073e-01],
       [1.26764167e-02, 9.87323583e-01],
       [9.68861410e-01, 3.11385903e-02],
       [1.12635271e-02, 9.88736473e-01]])

**Calculate the score using cross validation**

In [45]:
from sklearn.model_selection import cross_val_score
cross_val_score(GaussianNB(),X_train, y_train, cv=5)

array([0.824     , 0.76      , 0.776     , 0.75806452, 0.75      ])